In [1]:
 using JuMP, Ipopt , Printf

In [2]:
l = 1.0 
dy = 1.0 
dx = 0.0 
epsilon = 0.1 
a = 60.0 
k = 10.0
th_cri = pi / 2 

1.5707963267948966

In [3]:
model = Model(Ipopt.Optimizer) 
set_silent(model)

In [4]:
#Variable
@variable(model , x_1 ) # x (spostamento)
@variable(model , x_2 ) #Theta 
@variable(model , x_3 ) #\dot{x}
@variable(model , x_4 ) #\dot{theta}

#Starting Values
set_start_value(x_1, 2.0)
set_start_value(x_2, 0.0)
set_start_value(x_3, 0.0)
set_start_value(x_4, 0.0)

In [5]:
#Model Constrain
function b_0(x_1,x_2)
    return x_1^2+l^2+dx^2+dy^2-2*(x_1*l*sin(x_2)+x_1*dx-dx*l*sin(x_2)+dy*l*cos(x_2) )- epsilon^2
end

function b_1(x_1,x_2,x_3,x_4)
    val_b0 = b_0(x_1,x_2)
    val_b1 = x_3*(2*x_2-2*l*sin(x_2)-2*dx)-2*x_4*(x_1*l*cos(x_2)-dx*l*cos(x_2)-dy*l*sin(x_2))
    return val_b1+k*val_b0
end
#registration
register(model, :barrier0, 2, b_0, autodiff=true)
register(model, :barrier1, 4, b_1, autodiff=true)

#Constrain refered to the Obstacle
@NLconstraint(model , c1 ,barrier0(x_1,x_2) >= 0 )
@NLconstraint(model, c2 , barrier1(x_1,x_2,x_3,x_4) >= 0 ) 
#Constrain referd to the critical angle
@constraint(model, c3 , th_cri^2 - x_2^2 >= 0)
@constraint(model, c4 ,-2*x_4*x_2+a*(th_cri^2-x_2^2)>= 0)
#Objective Function
@objective(model,Min,x_1^2+x_2^2+x_3^2+x_4^2) #Paraboloid in 4 dimensions
print(model)


In [6]:
optimize!(model)
@printf("The solution is (x , θ , ẋ , θ̇ ) =(%.3f , %.3f , %.3f  %.3f ) " , value(x_1) , value(x_2) , value(x_3) , value(x_4))   
is_solved_and_feasible(model)



******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

The solution is (x , θ , ẋ , θ̇ ) =(0.050 , -0.050 , -0.000  -0.000 ) 

true